# Run a round of the adaptive DoE workflow

## What this workflow is
Adaptive **design of experiments (DoE)** is a way to find the best **recipe** — the combination of
settings (temperature, moisture, strain, substrate, …) that maximises a result you care about
(yield, biomass, colonisation speed, …) — **without** testing every possible combination. Rather
than running one giant grid of experiments, or guessing by trial and error, you run a **small
round at a time**. After each round a statistical model learns from your measured results and
proposes the **next, smarter batch**: it concentrates effort where the payoff looks highest while
still probing regions it is unsure about. Over a handful of rounds it closes in on a strong recipe
using far fewer experiments than a full grid would need.

## The loop — and where this notebook fits
The workflow is a **closed loop**:

> **propose** a round of recipes → **run** them at the bench → **record** the results → the model
> **learns** → **propose** the next round → …

**This notebook is one turn of that loop.** You run it start to finish to get one round, record its
results, and see where things stand — then, for the next round, you run it again. Everything
accumulates in a **single campaign file** (one CSV), so the model always learns from *all* of your
results so far, round after round. That file is also your complete, honest record of the campaign.

## What you provide, and what the workflow handles
- **You provide:** a description of your experiment — the factors you can change and their ranges,
  and the **one response** you want to optimise (step 1) — plus your measured results after each
  round (step 6).
- **The workflow handles:** fitting the model, choosing a diverse and promising set of recipes,
  producing a printable run sheet, keeping a tamper‑resistant record, and reporting **how much it
  has actually learned** so you know whether to trust its recommendation.

You do **not** need a statistics background: every number it reports is a physical measurement or a
percentage, and every figure carries a plain‑language caption.

## The seven steps at a glance
| Step | What happens |
|---|---|
| **Settings** | Set this round's knobs: how many recipes, how many replicates, and the campaign file. |
| **1. Describe your experiment** | Define your factors, units, and the response to optimise (the *domain*). |
| **2 & 3. Propose the conditions** | The model picks this round's recipes (an even, space‑filling spread on the very first round). |
| **4. Print the run sheet** | Get a printable sheet plus a blank results file to take to the bench. |
| **5 & 6. Run, then record** | Run the round, measure the results, and record them into the campaign. |
| **7. Look at the status page** | See the best recipe so far and honest checks on whether to trust it. |

## Demo vs. real use — read this before you run
As written, this notebook runs **top to bottom as a self‑contained demo**: a fermentation
**simulator stands in for your bench** at steps 5–6, so results appear instantly and the whole loop
executes in one go. In **real use**, two things are different:

1. **There is a real pause at steps 5–6.** You print the run sheet, actually run the round — days
   or weeks for a living culture — and only return to record once your measurements are in hand.
2. **You delete the demo scaffolding**, as flagged in the cells: the `LEDGER.unlink(...)` line in
   *Settings* (it wipes the campaign clean on every run, which you do **not** want for a real
   ongoing campaign) and the "DEMO BENCH" cell in step 5 (you fill the results file yourself
   instead).

*(The outputs shown in this notebook are from the demo run; your real numbers will differ.)*

## What this workflow can't do
- **One response at a time** — no multi‑objective optimisation. If you also care about, say,
  contamination, run yield as the response and watch contamination as the **failure rate**.
- **No *native* mixture designs** — you can't declare "components sum to a fixed total"
  directly, but you can still handle it: vary `n − 1` of the components and bound the remainder
  with a `<=`/`>=` constraint (see *Mixture / fixed‑total components* in step 1).
- **It never tells you when to stop** — it reports what it knows; **you** decide when the recipe is
  good enough. (See the README for the full list of limits.)

## How each step below is laid out
Every step tells you three things: the **tuning options** you can change (and how to choose them),
**what the step does and why**, and **how to read its output**. The knobs you touch routinely live
in the **Settings** cell and in **step 1** (your domain); the rest run as‑is. The rhythm is simple:
read the short explanation, run the cell, check the output, move on.

## Settings — edit these, then run every cell

**Tuning parameters (the knobs you set each round):**

| Parameter | What it controls | How to choose it |
|---|---|---|
| `CONDITIONS` | how many recipes this round, **including 1 control** | more = broader coverage per round but more bench work; 8–16 is typical |
| `REPLICATES` | replicates per recipe | `1` to explore cheaply; raise to `2`–`3` when you want to pin down noise or confirm a winner |
| `SEED` | reproducibility | leave fixed so a rerun reproduces the same plan; change it only to draw a different random proposal |
| `LEDGER` | the campaign file (one CSV, one row per replicate) | one file per campaign; keep appending to it round after round |

**What this does:** loads the library and sets the four knobs above. The `LEDGER.unlink(...)`
line resets the demo to a clean campaign on every run — **delete it for real work**, or you would
wipe your own history.

**How to read the output:** this cell prints nothing; if it runs without error, you are set up.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")

# Make the `adoe` package importable even without `pip install -e .`: put the
# repo's src/ on the path. (If you installed the package, this is a harmless no-op.)
_ROOT = Path.cwd()
if not (_ROOT / "pyproject.toml").exists():
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from adoe import (
    initial_design, propose, confirm, save_proposal, record, read_ledger,
    status, run_sheet, default_fermentation_domain,
)
from adoe.simulate import FermentationSimulator   # DEMO bench only (used in steps 5-6)

# adoe.report selects matplotlib's Agg backend on import; restore inline so the
# status figure in step 7 renders inside the notebook.
%matplotlib inline

# ---- TUNING PARAMETERS: the knobs you set each round ----
SEED = 0                             # reproducibility; same seed -> same proposal
LEDGER = Path("my_campaign.csv")     # one CSV holds the whole campaign (one row per replicate)
CONDITIONS = 12                      # recipes this round (INCLUDES 1 control)
REPLICATES = 1                       # replicates per recipe (1 = explore; 2-3 = pin down noise / confirm)

# DEMO ONLY: start each run from a clean campaign. DELETE this line for real work.
#LEDGER.unlink(missing_ok=True)

## 1. Describe your experiment

This is where you tell the workflow **what you are optimizing and what you are allowed to
change** — the *domain*. It is the single most important cell to get right: everything
downstream (the proposals, the run sheet, the status page) is expressed in the units you
declare here, and all values are **real, physical units** — the workflow handles the internal
maths for you.

The code cell below is a full, copy‑paste `Domain(...)` template. This section explains what
each part means, the options, and the rationale, so you can fill it in with intent rather than
by guesswork.

### Continuous factors — numeric settings you dial
Anything you set to a **number**: temperature, moisture, a concentration, incubation time. Each
is a `ContinuousFactor(name=, group=, low=, high=, unit=, step=)`.

- **`low` / `high`** — the range you are willing to run: the search box. Proposals *never* fall
  outside it, so set it to the full span you would genuinely try. Too wide wastes budget
  exploring settings you already know are poor; too narrow can fence the optimum out. When in
  doubt, err slightly wide — the loop concentrates effort where it matters.
- **`step`** — the finest increment your equipment can actually achieve (e.g. `0.5` °C, `1` %).
  Every proposed setpoint is **rounded to this grid**, so you never get "incubate at 31.4137 °C".
  Set it to the real resolution of your instrument or protocol.
- **`unit`** — the physical unit (`"°C"`, `"%"`, `"g/L"`, `"days"`). Shown on the run sheet; for
  you and record‑keeping only.
- **`group`** — a free‑text label used **only** as a section heading on the run sheet
  (e.g. `"substrate"`, `"process"`). No effect on the model — use whatever helps the bench read
  the sheet.
- **Holding a factor fixed:** set `low == high`. It is printed on every sheet but not searched
  or optimized — useful for documenting a constant you are deliberately not varying this campaign.

### Categorical factors — unordered choices
Options with **no natural numeric order**: strain, substrate type, supplier, vessel. Each is a
`CategoricalFactor(name=, group=, levels=[...])`.

- **`levels`** — the discrete options to compare (`["A15", "B22", "C07"]`). They are treated as
  **unordered and equally different**: the model learns each level's behaviour on its own rather
  than assuming "B sits between A and C".
- **How many:** comfortable at **2–6** levels; **data‑hungry beyond ~8**, because every level
  needs its own observations before the model can say anything about it. Each extra categorical
  factor also *multiplies* the number of combinations — the workflow refuses a campaign with more
  than 100 categorical combinations and will ask you to fix some for now.

### Block — the nuisance you cannot avoid but do not want to optimize
A **grouping** that carries hidden effects: the incubator or shelf used, the week it was run,
the lot of substrate. Declared with `block_column="block"`.

- **Why it exists:** experiments in the same block share unseen conditions (a warm incubator, a
  good substrate lot). Left unaccounted, those shared effects can **masquerade as real factor
  effects** — you would credit "strain B" for what was really "the good week". The workflow models
  the block as a nuisance and **controls for it**, and it **randomizes run order within the
  block**. Together that is the classical DoE discipline that stops slow drift and
  batch‑to‑batch differences from biasing your conclusions.
- **What to do:** leave it as `"block"`. The workflow assigns **one block per round**
  automatically (round 0 → block 0, round 1 → block 1, …), which matches how you actually work —
  a fresh batch each round. Set it to `None` only if you truly have no such grouping (rare in a
  real lab).

### Constraints — optional limits linking continuous factors
Linear limits on **combinations** of continuous settings, e.g. "two additives together must not
exceed a cap". Each is `LinearConstraint(columns=, coefficients=, sense=, rhs=)`, read as
`sum(coefficients × columns)  sense  rhs`.

- **Options:** only **`"<="`** or **`">="`** — there is no `"=="`. A "must sum to *exactly* a
  total" constraint is therefore not written directly; see **Mixture / fixed‑total components**
  just below.
- **Example:** `columns=["moisture","spawn_rate"], coefficients=[1,1], sense="<=", rhs=85` means
  `moisture + spawn_rate ≤ 85`; proposals are kept inside that region.
- **When to use:** a genuine operational or physical limit that *couples* factors. If your factors
  are independent, you usually need none — leave `linear_constraints=[]`.

### Mixture / fixed‑total components (media and substrate blends)
When several components must **add up to a fixed total** — a medium whose ingredients sum to
100 %, or a substrate blend that always fills the same mass — you **cannot** write
`carbon + nitrogen + filler == 100` directly: the workflow has no equality constraint, and a true
fixed‑sum "mixture" space needs special sampling this simplified version omits. Handle it with a
standard reparameterization, which the inequality constraints above support:

1. **Vary `n − 1` of the components** as ordinary continuous factors.
2. **Leave the last component out** — it is the **remainder** you compute yourself
   (`total − the others`), not a factor.
3. **Add one inequality** so that remainder stays in range.

**Example — a medium of three parts that must total 100 %** (a carbon source, a nitrogen source,
and a filler that makes up the balance):

```python
continuous=[
    ContinuousFactor(name="carbon",   group="media", low=10, high=70, unit="%", step=1.0),
    ContinuousFactor(name="nitrogen", group="media", low=5,  high=40, unit="%", step=0.5),
    # 'filler' is NOT a factor -- you set filler = 100 - carbon - nitrogen when you mix it
],
linear_constraints=[
    # keep the filler at or above 0 %:  carbon + nitrogen <= 100
    LinearConstraint(columns=["carbon", "nitrogen"], coefficients=[1, 1], sense="<=", rhs=100),
],
```

When you prepare each recipe, set `filler = 100 − carbon − nitrogen`. **Nothing is lost:** because
the filler is fully determined by the other two, varying `n − 1` components explores the entire
mixture. Record only the `n − 1` factors; the filler follows from them.

**Bounding the remainder** (optional) — put limits on the implied component with more inequalities:
- filler **at least** 20 % → `carbon + nitrogen <= 80`
- filler **at most** 60 % → `carbon + nitrogen >= 40`

If your components' own ranges already cannot exceed the total (e.g. two ingredients each capped
well below 100 %), you may need no constraint at all — add one only when the remainder could
otherwise go out of range.

### Target — the one response you are optimizing
The single measured outcome the workflow tries to improve, via
`TargetSpec(name=, unit=, direction=, transform=, delta_practical_pct=)`.

- **One response only.** No multi‑objective. If you also care about, say, contamination, do not
  add it as a second target — run yield as the target and **watch contamination as the failure
  rate** (mark contaminated replicates as failed when you record results).
- **`direction`** — `"maximize"` for a good thing (yield, biomass) or `"minimize"` for a bad
  thing (days to colonize, contamination rate).
- **`transform` — how the response is modelled.** Worth understanding:
  - **Why it matters:** a biological yield's noise **scales with its level** — a 500 g flush
    varies more in grams than a 50 g flush, but by roughly the same *percentage*. Modelling
    `log(yield)` turns that proportional variation into approximately **constant** noise. That is
    what lets the workflow report noise as a single **percentage** (±X %), express the smallest
    worthwhile gain as a percentage, and give intervals that are correct on the ratio scale.
  - **`"log"` (default)** — for a **positive response with proportional variation**: yield,
    biomass, titer, biological efficiency. A non‑positive value (`y ≤ 0`) is recorded as a **dead
    replicate / failure**, never as a small number.
  - **`"none"`** — for a **genuinely additive response that can be zero or negative**: a
    temperature, a pH, a percentage that can hit zero, a signed difference. The response is
    modelled directly, and noise and targets are in the **raw unit** rather than percentages.
  - **Rule of thumb:** a "how much did we make" quantity that is always positive and whose spread
    grows with its size → `"log"`; a physical level or a difference on an additive scale →
    `"none"`. When unsure, `"log"` is the right default for yields.
- **`delta_practical_pct`** — the **smallest improvement worth acting on**, as a percent of the
  current level (e.g. `10.0` = 10 %). It is **not a target to hit** — it is the *resolution you
  care about*. It drives the "roughly how many replicates to detect a gain this size" guide on the
  status page, so set it to the smallest gain that would actually change a decision.

### Before you move on
- Every **factor name must be unique**, and `block_column` must not reuse a factor name.
- **Read the printed `describe()`** below the template — it echoes the exact space the workflow
  will search: your ranges, steps, units, the block, any constraints, and the
  practical‑improvement threshold. Confirm it matches your real equipment and intent **before**
  proposing anything.

In [ ]:
# ============================================================================
#  HOW TO DEFINE YOUR OWN EXPERIMENT  --  copy this template, delete the
#  leading "# " on each line, and edit the values. Everything is in real units.
#  Then it replaces `domain = default_fermentation_domain()` at the bottom.
# ============================================================================
#
# from adoe import (Domain, ContinuousFactor, CategoricalFactor,
#                   TargetSpec, LinearConstraint)
#
# domain = Domain(
#
#     # --- CONTINUOUS FACTORS: numeric settings you dial -----------------------
#     #   name  = column name used everywhere
#     #   group = a heading on the run sheet (any word; e.g. substrate/process)
#     #   low/high = the range you are willing to run, in `unit`
#     #   step  = finest increment your equipment can actually set
#     #           (every proposed setpoint is rounded to this grid)
#     continuous=[
#         ContinuousFactor(name="moisture",    group="substrate", low=55, high=70, unit="%",  step=1.0),
#         ContinuousFactor(name="spawn_rate",  group="substrate", low=5,  high=20, unit="%",  step=0.5),
#         ContinuousFactor(name="temperature", group="process",   low=20, high=28, unit="°C", step=0.5),
#     ],
#
#     # --- CATEGORICAL FACTORS: unordered choices ------------------------------
#     #   levels = the options to compare (2-6 is comfortable; data-hungry beyond ~8)
#     categorical=[
#         CategoricalFactor(name="strain",    group="strain",    levels=["A15", "B22", "C07"]),
#         CategoricalFactor(name="substrate", group="substrate", levels=["straw", "sawdust"]),
#     ],
#
#     # --- BLOCK: the nuisance grouping (incubator / week / batch) --------------
#     #   Modeled and controlled for, never optimized. Leave as "block" (the
#     #   workflow assigns one block per round). Set to None only if you truly
#     #   have no such grouping.
#     block_column="block",
#
#     # --- CONSTRAINTS: optional linear limits on continuous factors ------------
#     #   Only "<=" or ">=". Use [] for none. To enable the example below, delete
#     #   the extra "# " in front of the LinearConstraint line. This one keeps
#     #   moisture + spawn_rate at or under 85% combined.
#     linear_constraints=[
#         # LinearConstraint(columns=["moisture", "spawn_rate"], coefficients=[1, 1], sense="<=", rhs=85),
#     ],
#
#     # --- TARGET: the ONE response you are optimizing --------------------------
#     target=TargetSpec(
#         name="yield",              # what you measure
#         unit="g",                  # its unit
#         direction="maximize",      # "maximize" a yield, or "minimize" a bad thing (days, contamination)
#         transform="log",           # "log" for a yield/biomass (noise grows with level);
#                                    #   use "none" only for an additive response that can be 0 or negative
#         delta_practical_pct=10.0,  # smallest improvement worth chasing, as a PERCENT
#     ),
# )
# ============================================================================

domain = default_fermentation_domain()   # <-- built-in demo domain; replace with your own (template above)
print(domain.describe())   # sanity-check the searched space, units, and step grid

## 2 & 3. Propose this round's conditions

**What this does, and why.** `propose` looks at everything recorded so far, fits a model to it,
and chooses this round's recipes. It works in two regimes:

- **Round 0 (cold start):** there is nothing to fit yet, so it lays down an even
  **space‑filling** design — a spread of recipes across your whole domain, balanced across the
  categorical levels (strains, substrates) — to give the model a broad first look.
- **Later rounds:** it fits a Gaussian‑process model to your recorded results and picks recipes
  that balance **pushing on the current best** (exploitation) against **covering regions it is
  unsure about** (exploration). Recipes are selected one at a time, each aware of the others
  already chosen this round, so you get a *diverse* batch rather than a dozen near‑duplicates.

Every round it also includes the **control** (your fixed reference recipe), and — crucially — it
**writes each recipe's predicted yield into the ledger now, before you run it**. That
commit‑before‑you‑run step is what makes the honesty checks in step 7 fair: the model cannot
grade itself after seeing the answer.

### The knobs you tune (set in the Settings cell)

- **`CONDITIONS` — how many recipes this round, including the 1 control.** More recipes = broader
  coverage per round and faster learning, but more bench work; fewer = cheaper rounds but slower
  progress. A practical range is **8–16**. One recipe is always the control, so `CONDITIONS=12`
  means 11 model‑chosen recipes plus the control.
  - *Consideration — make round 0 big enough.* The very first (space‑filling) design should have
    at least about **3 recipes per categorical combination** and **2 × (number of continuous
    factors + 1)** recipes overall, or the model starts nearly blind. The cell prints a warning if
    your round‑0 size looks too small — heed it, or expect over‑confident early proposals.
- **`REPLICATES` — replicates per recipe (`r`).** Use **`1`** early, to explore cheaply and cover more
  ground. Raise to **`2`–`3`** when you want to **measure repeat‑to‑repeat noise** or **confirm a
  promising recipe** — replicates buy precision at the cost of distinct recipes. Total replicates =
  `CONDITIONS × REPLICATES`, so at a fixed bench budget more replicates means fewer distinct
  recipes. (There is no need to replicate everything: 1 replicate is fine until a winner emerges,
  then confirm it — see step 7.)
- **`SEED` — reproducibility.** The same seed always produces the same plan. Change it only if you
  deliberately want a *different* random draw (e.g. a fresh space‑filling layout on round 0).

### Advanced knobs (passed straight to `propose`, rarely needed)

- **`control=`** — supply your own reference recipe as a `{factor: value}` dict. If you omit it,
  round 0 falls back to the domain midpoint (with a warning) and later rounds reuse whatever
  control is already in the ledger. Set it once to something meaningful — e.g. your current
  production recipe — so the "**% over control**" on the status page is a number you actually care
  about.
- **`num_restarts=`, `raw_samples=`** — how hard the optimiser searches for each next recipe.
  Higher is slightly more thorough but slower; the defaults are fine for normal bench use.
- **`capacity=`** (instead of `n_conditions`) — give a total **replicate budget** and let it solve
  `n_conditions = capacity // r`, reporting any leftover replicates.
- **`block=`** — normally leave it alone; the workflow labels each round with its own block
  automatically (see the block discussion in step 1).

### How to read the output (the table below)

- **One row per replicate.** `execution_order` is the order to run them at the bench (randomized on
  purpose, so slow drift over the day cannot masquerade as a real factor effect).
- **`slot_type`** is `explore` (a model‑chosen best guess) or `control` (the fixed reference).
- Replicates of the same recipe share a `condition_id` but get distinct `unit_id`s.
- Each recipe also carries a **locked‑in predicted yield** in the ledger (not shown in this short
  preview) that cannot be edited afterwards — the write‑once discipline behind step 7.
- Nothing is on the bench yet: `save_proposal` has appended this round to your campaign CSV so
  that `record` (step 6) can match your measured results back to these exact recipes.

In [ ]:
# 1) Load the campaign so far. Empty on round 0 -> propose() cold-starts with a
#    space-filling design; on later rounds it fits a model to what you've recorded.
ledger = read_ledger(LEDGER)

# 2) Propose this round's recipes.
#    Required knobs (from the Settings cell):
#        n_conditions = CONDITIONS   # recipes this round, INCLUDING the 1 control
#        r            = REPLICATES   # replicates per recipe
#        seed         = SEED         # reproducible plan
#    Optional knobs (add them to the call to use):
#        control={<factor>: <value>, ...}  # your own reference recipe
#        num_restarts=..., raw_samples=... # harder search (slower; defaults are fine)
#        capacity=<total replicates>             # budget by total replicates instead of n_conditions
run = propose(ledger, domain, n_conditions=CONDITIONS, r=REPLICATES, seed=SEED)

# 3) Append this round -- and its write-once predictions -- to the campaign CSV.
save_proposal(LEDGER, run)

# 4) Preview: one row per replicate, in execution order. Full details (incl. the locked-in
#    predictions) live in the ledger CSV, not in this short preview.
run[["execution_order", "unit_id", "slot_type", *domain.factor_names]].head(len(run))

## 4. Print the run sheet

**What this does.** Turns this round's proposal into two files you use at the bench:

- **`round.html`** — a **printable run sheet**. It lists every replicate to run, its recipe in
  your real units, and the order to run them, with a summary of the experiment in the header.
  Open it in a browser and print it (or keep it on a tablet at the bench).
- **`round.results.csv`** — a **blank data‑entry file**: one row per replicate, with the
  identifying details already filled in and an empty `y` column for the result. You type your
  measurements into this file, and step 6 reads it back.

**The one thing you can change.** The name `"round"` sets the two filenames (`round.html` and
`round.results.csv`). Running the cell again overwrites them. If you'd rather keep a separate
record for each round, give each its own name — for example
`run_sheet(run, domain, f"round{n}")` for round number `n`.

**How to use it at the bench.**

1. Print `round.html` (or open it on a device).
2. Run the replicates **in the order shown** — the `execution_order` column. That order is
   shuffled on purpose. If you always ran the best‑guess recipes first and the control last, a
   slow change over the day — a warming incubator, an aging inoculum — could make some recipes
   look better or worse than they really are. A randomised order spreads that drift evenly across
   the recipes, so it can't bias the comparison.
3. As results come in, type each measured value into the **`y` column** of the results CSV (and
   flag anything that failed — see step 6).

Nothing is scored yet: you are just carrying the plan to the bench and collecting the numbers.

In [ ]:
# "round" is the output base name -> round.html (print this) + round.results.csv (fill this in).
html_path, results_path = run_sheet(run, domain, "round_00")
print("run sheet:", html_path)      # print and take to the bench; run in execution order
print("fill in:  ", results_path)   # type each measured yield into the blank y column here

## 5 & 6. Run the experiment, then record the results

**When to run this — and why the gap matters.** This is the one step that does not happen at the
keyboard. After step 4 you take the run sheet to the bench, **run the round, and wait for
results** — which for a mycelial culture can be days or weeks. Only once the round is finished and
you have typed the measurements into `round.results.csv` do you come back and record them.

> In this demo the two halves run back‑to‑back because a simulator stands in for the bench. In
> real work there is a genuine pause here: **stop after step 4, go run the experiment, then return
> to this step** with your filled‑in results file.

**What recording does.** `record()` reads your filled results CSV and writes the outcomes — the
result for each replicate, plus any failures — into the campaign ledger (your single CSV). That
is what makes this round's results available to the next round's model (step 2) and to the status
page (step 7).

**Why there is a strict rule.** `record()` will **refuse** a results file in which any setpoint or
any stored prediction has been changed — those columns are **write‑once**. The reason is fairness:
back in step 2 the model wrote down its prediction for each recipe *before* you ran it, and the
honesty checks in step 7 compare those predictions against what actually happened. If predictions
or recipes could be edited after the fact, the model could quietly grade its own homework. So:
fill in your results, and leave everything else exactly as it was.

**Entering your data** (only ever the outcome columns):

- **A normal result** — type the measured value into the **`y` column**.
- **A failure** (contaminated, died, no growth) — set **`failed` to `True`** and leave `y` blank.
  A non‑positive result is treated as a failure automatically (recorded as "no yield"). You can
  add a note in `failure_reason` if you like.

**The demo cell below** simulates the bench so this notebook can run end‑to‑end. **Delete it for
real work** — instead fill `round.results.csv` yourself, and then the single line
`record(LEDGER, "round.results.csv")` is all you need.

**What you'll see:** `recorded N units` confirms the round is now in the ledger. From here you can
look at the status page (step 7), or go back to step 2 to propose the next round.

In [ ]:
# ============== DEMO BENCH — DELETE THIS BLOCK FOR REAL WORK ==============
# Stands in for the lab by simulating yields into the results CSV. For real
# work, fill round.results.csv by hand and skip straight to record() below.
#sim = FermentationSimulator()
filled = pd.read_csv(results_path)
#frame = filled[domain.factor_names].copy()
#frame["block"] = filled["block"].to_numpy()
#outcome = sim.evaluate(frame, np.random.default_rng(SEED))
#filled["y"] = np.where(outcome["failed"].to_numpy(), np.nan, outcome["y"].to_numpy())
#filled["failed"] = outcome["failed"].to_numpy()
#filled.to_csv(results_path, index=False)
# =========================================================================

# Comment out all lines in the Demo Bench block above EXCEPT 'filled'
# This one line is all you need for real work, once the CSV is filled in:
record(LEDGER, results_path)                 # write-once: rejects altered setpoints/predictions
print("recorded", len(filled), "units")

## 7. Look at the status page

This is where you read **how the campaign is going**. It is a **read‑out only** — it never tells
you to stop or continue; it lays out what is known and *you* decide. The printed lines give the
current best recipe; the figure has **four panels** (a 2×2 grid) covering the recipe, progress,
and two honesty checks.

Two terms are used throughout, worth pinning down first:

- **Model incumbent** — the single recipe the *fitted model* currently believes is best: its top
  recommendation. It is found by searching the model's predicted surface, so it can be a recipe
  you have **not actually run yet**.
- **Best observed** — the best result you have **actually measured** so far. As the campaign
  matures these two should converge; a large gap means the model's pick still needs a real run to
  confirm it.

### Top‑left — Best recipe (the model incumbent)
- **What it shows:** the recommended setpoint for every factor, its **predicted typical yield**
  with a **95% interval**, and its **improvement over the control** as a percent.
- **Why it's here:** this is your current answer — the recipe to try or scale next.
- **How to read it:** "predicted typical" is a **median**, not an average — because the model
  works on `log(yield)`, so about half of identical replicates would land above this number and
  half below. The **95% interval** is the range a single new replicate should fall in about 95% of
  the time — *provided the error bars are honest* (check B). "**vs control: +X%**" is how much
  better this recipe is predicted to be than your fixed reference.

### Top‑right — Progress
- **What it shows:** a line of the **best result observed so far** against the number of successful
  runs completed (cumulative), with a dashed line at the **model incumbent's** predicted yield.
- **Why it's here:** it answers "are we getting somewhere, and how fast?"
- **How to read it:** a curve that **rises then flattens** means you are converging — extra rounds
  are buying less. Still **climbing steeply** means keep going. **Flat from the start** means
  nothing is improving (check A tells you whether that is because the model has not learned). If
  the dashed model‑incumbent line sits well **above** the best you have actually observed, the
  model is pointing at a recipe you have not confirmed — a good moment to run a confirmation round.

### Bottom‑left — Honesty check A: is it learning?
- **What it shows:** two bars — the **model's prediction error** vs the error of **just guessing
  the average** ("guess‑the‑mean"), both on the log scale; **lower is better**.
- **Why it's here:** a model no better than guessing should not be trusted to recommend recipes.
  This is the guardrail against false confidence.
- **How to read it:** **model bar much lower** = the model is learning real structure from your
  factors, so its recommendations mean something. **Bars roughly equal** = it has not learned yet
  — normal in the first round or two, but if it persists, either you need more data or the factors
  you chose do not move the response. Treat recommendations skeptically until this bar drops.

### Bottom‑right — Honesty check B (error bars), noise, and failures
- **`rms_z` (target ≈ 1.0) — are the 95% intervals the right size?** For each recorded result, its
  gap from the prediction is divided by that prediction's uncertainty; `rms_z` combines those into
  one number. **≈ 1** = honestly sized. **Above ~1.25** = the intervals are **too narrow /
  overconfident** (the panel flags this) — common early on; treat the recipe card's interval as
  rough and confirm before committing. **Well below 1** = intervals wider than necessary. It uses
  RMS rather than a plain spread precisely so it catches a *consistent* bias, not just scatter.
- **95% coverage (target ≥ 90%):** the share of results that actually landed inside their 95%
  intervals. Below 90% is the same warning as a high `rms_z`.
- **Repeat‑to‑repeat noise (±%):** how much identical replicates vary — your process's built‑in
  variability. This is the bar an improvement has to clear to be real, not noise.
- **Worthwhile gain / rough replicates to detect it:** the smallest improvement you said matters
  (`delta_practical_pct`) and roughly how many replicates you would need to tell a gain that size
  apart from the noise. Use it to decide how hard to replicate a candidate before believing it.
- **Failure rate** (shown only if some runs failed): the share of runs that failed. A rising rate
  often means you are pushing into a hostile region (e.g. extreme temperature or pH).

### What to do next — you decide (the page won't)
- **Still learning and progress climbing** → run **another round**: go back to step 2 (remove the
  `LEDGER.unlink` line in Settings first, so the campaign keeps growing instead of resetting).
- **A clear front‑runner, check A good, but check B shaky (or few replicates)** → **confirm it**:
  `confirm(ledger, domain, r=3)` replicates the best recipe against the control so you can trust
  the comparison before scaling.
- **Not learning after a few rounds** → revisit step 1: your ranges may be too narrow (the optimum
  fenced out) or too wide (budget spread thin), or the factors may not be the ones that matter.

In [ ]:
# The status page is a READ-OUT: it summarizes the campaign and never tells you to
# stop or continue -- you decide. See the markdown above for how to read each panel.
report = status(LEDGER, domain)

# --- printed: the current best recipe (the "model incumbent" -- the model's top pick) ---
print("Best recipe so far:")
for name in domain.factor_names:             # recommended setpoint for each factor, in real units
    print(f"  {name}: {report.best_recipe[name]}")
# "predicted typical" is a MEDIAN yield (about half of identical replicates land above it, half
# below), not an average -- because the model works on log(yield). The 95% interval is in the figure.
print(f"  predicted typical {domain.target.name}: {report.best_recipe['predicted_median']:.3g} {domain.target.unit}")
if report.improvement_pct is not None:       # predicted gain of this recipe over your control, as a %
    print(f"  improvement over control: {report.improvement_pct:+.0f}%")

# --- the four-panel figure:  recipe | progress | check A (is it learning?) | check B (honest bars?) ---
import matplotlib.pyplot as plt
plt.close(report.figure)                     # (keeps the notebook from rendering the figure twice)
report.figure